# Unidad 1 · Prueba de hipótesis: valor p, región de rechazo y errores

Reproduce el ejemplo resuelto (sacos de pellets) y la pieza del caso integrador (pulpo 6 contra la meta de
0,10 mm). Lee los datos de `../datos/`. Cada prueba sigue los cinco pasos: pregunta, hipótesis, estadístico,
valor p o región de rechazo y conclusión.

In [ ]:
from pathlib import Path
from math import sqrt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.power import NormalIndPower

DATOS = Path("..") / "datos"
RES = {}

## 1. Ejemplo · ¿los sacos vienen livianos?

$H_0: \mu = 15$ kg contra $H_1: \mu < 15$ kg, con $\alpha = 0{,}05$.

In [ ]:
x = pd.read_csv(DATOS / "u01_pellets.csv").peso_kg.to_numpy()
n = len(x); xb = x.mean(); s = x.std(ddof=1); ee = s / sqrt(n)
t = (xb - 15) / ee                        # a mano
p = stats.t.cdf(t, n - 1)                 # cola izquierda
print(f"x̄ = {xb:.4f}  s = {s:.4f}  t = {t:.3f}  p = {p:.4f}")
print(stats.ttest_1samp(x, 15, alternative="less"))   # lo mismo con scipy
print("Shapiro-Wilk:", stats.shapiro(x))
ic = stats.t.interval(0.95, n - 1, loc=xb, scale=ee)
print("IC 95 %:", ic)
RES.update(ej_t=t, ej_p=p, ej_ic_lo=ic[0], ej_ic_hi=ic[1])

## 2. Potencia y tamaño de muestra (con $z$, tomando $\sigma \approx s$)

Potencia para detectar sacos 100 g livianos y $n$ para una potencia de 0,90.

In [ ]:
z = stats.norm.ppf
delta = 0.10
pot = stats.norm.cdf(delta / (s / sqrt(n)) - z(0.95))
n_nec = ((z(0.95) + z(0.90)) * s / delta) ** 2
print(f"potencia con n = {n}: {pot:.3f}   n para 0,90: {n_nec:.2f}")
# statsmodels: ratio=0 es la prueba z de una muestra
print(NormalIndPower().power(effect_size=delta / s, nobs1=n, alpha=0.05, ratio=0, alternative="larger"))
RES.update(ej_pot=pot, ej_n=n_nec)

## 3. Caso · ¿el pulpo 6 supera la meta de 0,10 mm?

$H_0: \mu = 0{,}10$ contra $H_1: \mu > 0{,}10$.

In [ ]:
reg = pd.read_csv(DATOS / "registro.csv")
a = reg[reg.maquina == "pulpo 6"].desplazamiento_mm.to_numpy()
b = reg[reg.maquina == "pulpo 4"].desplazamiento_mm.to_numpy()
print("Shapiro-Wilk pulpo 6:", stats.shapiro(a))
r6 = stats.ttest_1samp(a, 0.10, alternative="greater")
ic6 = stats.t.interval(0.95, len(a) - 1, loc=a.mean(), scale=a.std(ddof=1) / sqrt(len(a)))
print(f"pulpo 6: x̄ = {a.mean():.4f}  t = {r6.statistic:.2f}  p = {r6.pvalue:.3f}  IC 95 % = [{ic6[0]:.3f}; {ic6[1]:.3f}]")
s6 = a.std(ddof=1)
pot6 = stats.norm.cdf(0.03 / (s6 / sqrt(len(a))) - z(0.95))
n6 = ((z(0.95) + z(0.80)) * s6 / 0.03) ** 2
print(f"potencia para detectar 0,13 mm: {pot6:.3f}   n para potencia 0,80: {n6:.1f}")
r4 = stats.ttest_1samp(b, 0.10, alternative="greater")
print(f"pulpo 4: x̄ = {b.mean():.4f}  t = {r4.statistic:.2f}  p = {r4.pvalue:.2e}")
RES.update(caso_media=a.mean(), caso_t=r6.statistic, caso_p=r6.pvalue, caso_ic_lo=ic6[0], caso_ic_hi=ic6[1],
           caso_pot=pot6, caso_n=n6, caso4_t=r4.statistic, caso4_p=r4.pvalue)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 2.6))
ax.hist(a, bins=np.arange(0, 0.26, 0.02), alpha=.5)
ax.axvline(0.10, color="tab:orange", lw=2, label="meta 0,10 mm")
ax.axvline(a.mean(), color="k", ls="--", label="media pulpo 6")
ax.set_xlabel("desplazamiento (mm)"); ax.legend(frameon=False)
plt.show()